In [ ]:
__nbid__ = '0081'
__author__ = 'Seo-Won Chang and the KS4 Team'
__version__ = '20260223'
__datasets__ = ['ks4_dr1']
__keywords__ = ['star-galaxy separation', 'class_star', 'cross-match', 'Gaia']

# Source Classification Guidance & Pre-matched External Catalogs in KS4 DR1
_Seo-Won Chang and the KS4 Team_

### Table of contents
* [Goals & Summary](#goals)
* [Disclaimer & attribution](#attribution)
* [Imports & setup](#import)
* [1. class_star: The SExtractor Stellarity Index](#classstar)
  * [1.1 class_star vs magnitude](#classstar_mag)
  * [1.2 Band dependence of class_star](#classstar_band)
* [2. Cross-Matching with Gaia DR3](#gaia)
  * [2.1 Positional cross-match via Q3C](#gaia_xmatch)
  * [2.2 Astrometric comparison](#gaia_astrom)
* [3. Cross-Matching with Other Surveys](#other_xmatch)

<a class="anchor" id="goals"></a>
# Goals
* Understand the `class_star` stellarity index and its behavior across bands and magnitudes
* Perform positional cross-matches with Gaia DR3 and other catalogs using Q3C

# Summary
Building on the data access techniques introduced in Notebook 1 (*Getting Started with KS4 DR1*), this notebook focuses on source classification and multi-band photometry. We explore the SExtractor `class_star` parameter, demonstrate star-galaxy separation strategies, and show how to cross-match KS4 DR1 with external catalogs such as Gaia DR3.

<a class="anchor" id="attribution"></a>
# Disclaimer & attribution
If you use this notebook or KS4 DR1 data for your published science, please acknowledge the following:

* **KS4 Overview paper**: Im et al. (in preparation), *KMTNet Synoptic Survey of Southern Sky I: Overview*, Journal of the Korean Astronomical Society (JKAS)
* **KS4 DR1 pipeline paper**: Jeong et al. (2026), *KMTNet Synoptic Survey of Southern Sky II: Data Reduction and Real-Time Transient Detection Pipeline*, JKAS (accepted for publication)
* **KS4 DR1 paper**: Chang et al. (2026), *KMTNet Synoptic Survey of Southern Sky III: The First Data
Release*, JKAS (in revision)

Acknowledgments
---------------
If you use **Astro Data Lab** in your published research, please include the text in your paper's Acknowledgments section:

_This research uses services or data provided by the Astro Data Lab, which is part of the Community Science and Data Center (CSDC) Program of NSF NOIRLab. NOIRLab is operated by the Association of Universities for Research in Astronomy (AURA), Inc. under a cooperative agreement with the U.S. National Science Foundation._

If you use **SPARCL jointly with the Astro Data Lab platform** (via JupyterLab, command-line, or web interface) in your published research, please include this text below in your paper's Acknowledgments section:

_This research uses services or data provided by the SPectra Analysis and Retrievable Catalog Lab (SPARCL) and the Astro Data Lab, which are both part of the Community Science and Data Center (CSDC) Program of NSF NOIRLab. NOIRLab is operated by the Association of Universities for Research in Astronomy (AURA), Inc. under a cooperative agreement with the U.S. National Science Foundation._

In either case **please cite the following papers**:

* Data Lab concept paper: Fitzpatrick et al., "The NOAO Data Laboratory: a conceptual overview", SPIE, 9149, 2014, https://doi.org/10.1117/12.2057445

* Astro Data Lab overview: Nikutta et al., "Data Lab - A Community Science Platform", Astronomy and Computing, 33, 2020, https://doi.org/10.1016/j.ascom.2020.100411

If you are referring to the Data Lab JupyterLab / Jupyter Notebooks, cite:

* Juneau et al., "Jupyter-Enabled Astrophysical Analysis Using Data-Proximate Computing Platforms", CiSE, 23, 15, 2021, https://doi.org/10.1109/MCSE.2021.3057097

If publishing in a AAS journal, also add the keyword: `\facility{Astro Data Lab}`

And if you are using SPARCL, please also add `\software{SPARCL}` and cite:

* Juneau et al., "SPARCL: SPectra Analysis and Retrievable Catalog Lab", Conference Proceedings for ADASS XXXIII, 2024
https://doi.org/10.48550/arXiv.2401.05576

<a class="anchor" id="import"></a>
# Imports and setup

In [ ]:
# Standard library imports
from getpass import getpass
import warnings
warnings.filterwarnings('ignore')

# 3rd party
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
%matplotlib inline

# Data Lab
from dl import authClient as ac, queryClient as qc
from dl.helpers.utils import convert

# Plotting defaults
plt.rcParams.update({'font.size': 14, 'figure.facecolor': 'white'})

<a class="anchor" id="classstar"></a>
# 1. `class_star`: The SExtractor Stellarity Index

The `class_star_{band}` column is a neural-network-based classifier from SExtractor that returns a value between 0 (extended/galaxy) and 1 (point source/star). Its reliability depends on both the signal-to-noise ratio and the local seeing conditions.

<a class="anchor" id="classstar_mag"></a>
## 1.1 `class_star` vs magnitude

Let's first examine how `class_star` behaves as a function of magnitude. We query a sparse high-latitude field to minimize crowding effects.

In [ ]:
%%time
# Query a sparse field for star-galaxy separation analysis
query_sg = """
SELECT mag_auto_b, mag_auto_v, mag_auto_r, mag_auto_i,
       magerr_auto_b, magerr_auto_v, magerr_auto_r, magerr_auto_i,
       class_star_b, class_star_v, class_star_r, class_star_i,
       fwhm_image_i, kron_radius_i,
       flags_i, imaflags_iso_i, ssflag_i,
       ra, dec
FROM ks4_dr1.idual_master
WHERE q3c_radial_query(ra, dec, 50.0, -60.0, 0.5)
  AND flags_i = 0
  AND imaflags_iso_i = 0
  AND ssflag_i != -1
"""
res = qc.query(sql=query_sg)
df = convert(res, 'pandas')
print(f"Retrieved {len(df):,} clean sources.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

mask = df['mag_auto_i'].between(14, 25)
hb = ax.hexbin(df.loc[mask, 'mag_auto_i'], df.loc[mask, 'class_star_i'],
               gridsize=200, bins='log', mincnt=1, cmap='inferno')
ax.set_xlabel('mag_auto_i (AB)')
ax.set_ylabel('class_star_i')
ax.set_title('class_star_i vs I-band Magnitude')
ax.axhline(y=0.5, color='black', ls='--', lw=3.5, alpha=0.8, label='class_star = 0.5')
ax.axhline(y=0.8, color='cyan', ls='--', lw=3.5, alpha=0.8, label='class_star = 0.8')
ax.axhline(y=0.3, color='lime', ls='--', lw=3.5, alpha=0.8, label='class_star = 0.3')
ax.legend(loc='center left', fontsize=11)
plt.colorbar(hb, ax=ax, label='log(N)')
plt.tight_layout()
plt.show()

The plot above reveals the typical behavior of `class_star`:

* **Bright sources (I < 20)**: clear bimodal distribution — stars cluster near `class_star ~ 1` and galaxies near `class_star ~ 0`
* **Faint sources (I > 21)**: the two populations merge, making reliable classification increasingly difficult

<a class="anchor" id="classstar_band"></a>
## 1.2 Band dependence of `class_star`

The `class_star` value can differ between bands due to varying seeing and PSF shapes. Let's compare across B, V, R, and I bands.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, band, label in zip(axes, ['b', 'v', 'r', 'i'], ['B', 'V', 'R', 'I']):
    mag_col = f'mag_auto_{band}'
    cs_col = f'class_star_{band}'
    mask = df[mag_col].between(14, 25) & df[cs_col].notna()
    ax.hexbin(df.loc[mask, mag_col], df.loc[mask, cs_col],
             gridsize=150, bins='log', mincnt=1, cmap='inferno')
    ax.set_xlabel(f'mag_auto_{band}')
    ax.set_ylabel(f'class_star_{band}')
    ax.set_title(f'{label} band')
    ax.axhline(y=0.5, color='w', ls='--', lw=1, alpha=0.7)

plt.suptitle('class_star vs Magnitude in Each Band', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

<a class="anchor" id="gaia"></a>
# 2. Cross-Matching with Gaia DR3

Cross-matching with Gaia DR3 provides proper motions, parallaxes, and an independent photometric reference. Data Lab hosts the Gaia DR3 catalog, making SQL-based positional cross-matches straightforward.

<a class="anchor" id="gaia_xmatch"></a>
## 2.1 Positional cross-match via Q3C

In [ ]:
%%time
# Cross-match KS4 DR1 with Gaia DR3 within 1 arcsec
query_gaia = """
SELECT k.coadd_object_id, k.ra as ks4_ra, k.dec as ks4_dec,
       k.mag_auto_b, k.mag_auto_v, k.mag_auto_r, k.mag_auto_i,
       k.magerr_auto_i, k.class_star_i,
       g.source_id, g.ra as gaia_ra, g.dec as gaia_dec,
       g.phot_g_mean_mag, g.phot_bp_mean_mag, g.phot_rp_mean_mag,
       g.pmra, g.pmdec, g.parallax
FROM ks4_dr1.idual_master k
JOIN gaia_dr3.gaia_source g
  ON q3c_join(k.ra, k.dec, g.ra, g.dec, 0.00028)
WHERE q3c_radial_query(k.ra, k.dec, 50.0, -60.0, 0.3)
  AND k.flags_i = 0
  AND k.imaflags_iso_i = 0
  AND k.ssflag_i != -1
  AND k.mag_auto_i < 21
"""
res = qc.query(sql=query_gaia)
df_gaia = convert(res, 'pandas')
print(f"Cross-matched {len(df_gaia):,} sources (KS4 x Gaia DR3, within 1 arcsec).")

> 💡 **Pro Tip:** The `q3c_join` radius of `0.00028` degrees corresponds to ~1 arcsec.

<a class="anchor" id="gaia_astrom"></a>
## 2.2 Astrometric comparison

Let's check the positional offsets between KS4 and Gaia DR3.

In [ ]:
# Compute positional offsets in arcsec
df_gaia['dra'] = (df_gaia['ks4_ra'] - df_gaia['gaia_ra']) * 3600 * np.cos(np.radians(df_gaia['ks4_dec']))
df_gaia['ddec'] = (df_gaia['ks4_dec'] - df_gaia['gaia_dec']) * 3600

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 2D offset distribution
ax1.hexbin(df_gaia['dra'], df_gaia['ddec'],
          gridsize=100, bins='log', mincnt=1, cmap='inferno',
          extent=[-1, 1, -1, 1])
ax1.axhline(y=0, color='w', ls='--', lw=0.5, alpha=0.5)
ax1.axvline(x=0, color='w', ls='--', lw=0.5, alpha=0.5)
ax1.set_xlabel('ΔRA (arcsec)')
ax1.set_ylabel('ΔDec (arcsec)')
ax1.set_title('KS4 − Gaia DR3 Positional Offsets')
ax1.set_aspect('equal')

# Histogram of separations
sep = np.sqrt(df_gaia['dra']**2 + df_gaia['ddec']**2)
ax2.hist(sep, bins=100, range=(0, 1), histtype='step', lw=2, color='k')
ax2.set_xlabel('Separation (arcsec)')
ax2.set_ylabel('N')
ax2.set_title(f'Median separation: {sep.median():.3f} arcsec')
ax2.axvline(x=sep.median(), color='r', ls='--', label=f'Median = {sep.median():.3f}"')
ax2.legend()

print(f"Mean ΔRA:  {df_gaia['dra'].mean():.4f} ± {df_gaia['dra'].std():.4f} arcsec")
print(f"Mean ΔDec: {df_gaia['ddec'].mean():.4f} ± {df_gaia['ddec'].std():.4f} arcsec")
plt.tight_layout()
plt.show()

<a class="anchor" id="other_xmatch"></a>
# 3. Cross-Matching with Other Surveys

KS4 DR1 provides pre-computed cross-match tables with major external catalogs, matched at a 1.5 arcsec radius. These are available under the `ks4_dr1` schema and are much faster than performing `q3c_join` on the fly.

### Available cross-match tables

| Cross-match Table | External Catalog |
|---|---|
| `x1p5__idual_master__gaia_dr3__gaia_source` | Gaia DR3 |
| `x1p5__idual_master__allwise__source` | AllWISE |
| `x1p5__idual_master__unwise_dr1__object` | unWISE DR1 |
| `x1p5__idual_master__vhs_dr5__vhs_cat_v3` | VHS DR5 |
| `x1p5__idual_master__skymapper_dr4__master` | SkyMapper DR4 |
| `x1p5__idual_master__nsc_dr2__object` | NSC DR2 |
| `x1p5__idual_master__delve_dr2__objects` | DELVE DR2 |
| `x1p5__idual_master__delve_dr3__coadd_objects` | DELVE DR3 |

Equivalent tables exist for `single_master` (replace `idual_master` with `single_master` in the table name).

Each cross-match table contains the following columns:

| Column | Description |
|---|---|
| `id1` | KS4 identifier (`coadd_object_id`) |
| `id2` | External catalog identifier |
| `ra1`, `dec1` | KS4 coordinates (deg) |
| `ra2`, `dec2` | External catalog coordinates (deg) |
| `distance` | Angular separation between matched sources (arcsec) |

### Example: KS4 × AllWISE 

Let's use the pre-computed cross-match table to combine KS4 optical photometry with AllWISE mid-infrared photometry.

In [ ]:
%%time
# KS4 x AllWISE via pre-computed cross-match table
query_allwise = """
SELECT k.coadd_object_id, k.mag_auto_v, k.mag_auto_i,
       k.magerr_auto_i, k.class_star_i,
       w.w1mpro, w.w2mpro, w.w3mpro,
       xm.distance as xmatch_dist
FROM ks4_dr1.x1p5__idual_master__allwise__source xm
JOIN ks4_dr1.idual_master k ON xm.id1 = k.coadd_object_id
JOIN allwise.source w ON xm.id2 = w.source_id
WHERE q3c_radial_query(k.ra, k.dec, 50.0, -60.0, 0.3)
  AND k.flags_i = 0
  AND k.imaflags_iso_i = 0
  AND k.ssflag_i != -1
  AND k.mag_auto_i < 21
"""
res = qc.query(sql=query_allwise)
df_wise = convert(res, 'pandas')
print(f"KS4 x AllWISE: {len(df_wise):,} matches")
print(f"Match distance: median = {df_wise['xmatch_dist'].median():.3f} arcsec, "
      f"max = {df_wise['xmatch_dist'].max():.3f} arcsec")

In [ ]:
# Match distance distribution
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df_wise['xmatch_dist'], bins=100, range=(0, 1.5),
        histtype='step', lw=2, color='k')
ax.axvline(x=df_wise['xmatch_dist'].median(), color='r', ls='--',
           label=f"Median = {df_wise['xmatch_dist'].median():.3f}\"")
ax.set_xlabel('Match Distance (arcsec)')
ax.set_ylabel('N')
ax.set_title('KS4 × AllWISE Cross-Match Distance Distribution')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Optical-infrared color-color diagram: I - W1 vs W1 - W2
df_wise['i_w1'] = df_wise['mag_auto_i'] - df_wise['w1mpro']
df_wise['w1_w2'] = df_wise['w1mpro'] - df_wise['w2mpro']

mask_star = (df_wise['class_star_i'] > 0.9) & (df_wise['magerr_auto_i'] < 0.1)
mask_gal  = (df_wise['class_star_i'] < 0.1) & (df_wise['magerr_auto_i'] < 0.1)

fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(df_wise.loc[mask_gal, 'i_w1'], df_wise.loc[mask_gal, 'w1_w2'],
           s=1.5, alpha=0.8, c='red', label='Galaxies (class_star < 0.1)', rasterized=True)
ax.scatter(df_wise.loc[mask_star, 'i_w1'], df_wise.loc[mask_star, 'w1_w2'],
           s=1.5, alpha=0.8, c='blue', label='Stars (class_star > 0.9)', rasterized=True)

ax.set_xlabel('I$_{\\mathrm{KS4}}$ - W1')
ax.set_ylabel('W1 - W2')
ax.set_title('Optical-Infrared Color-Color: KS4 + AllWISE')
ax.set_xlim(-1, 8)
ax.set_ylim(-0.5, 2.0)
ax.legend(markerscale=10, fontsize=12)
plt.tight_layout()
plt.show()

### Example: KS4 × VHS DR5

The VISTA Hemisphere Survey (VHS) provides near-infrared JHKs photometry complementary to KS4's optical BVRI.

In [ ]:
%%time
# KS4 x VHS DR5 via pre-computed cross-match table
query_vhs = """
SELECT k.coadd_object_id, k.mag_auto_v, k.mag_auto_i,
       k.magerr_auto_i, k.class_star_i,
       v.japermag3 as j_mag, v.hapermag3 as h_mag, v.ksapermag3 as ks_mag,
       xm.distance as xmatch_dist
FROM ks4_dr1.x1p5__idual_master__vhs_dr5__vhs_cat_v3 xm
JOIN ks4_dr1.idual_master k ON xm.id1 = k.coadd_object_id
JOIN vhs_dr5.vhs_cat_v3 v ON xm.id2 = v.sourceid
WHERE q3c_radial_query(k.ra, k.dec, 50.0, -60.0, 0.3)
  AND k.flags_i = 0
  AND k.imaflags_iso_i = 0
  AND k.ssflag_i != -1
  AND k.mag_auto_i < 21
"""
res = qc.query(sql=query_vhs)
df_vhs = convert(res, 'pandas')
print(f"KS4 x VHS DR5: {len(df_vhs):,} matches")

> 💡 **Pro Tip:** The pre-computed cross-match tables use a 1.5 arcsec matching radius. To enforce a tighter match, add a filter on the `distance` column (e.g., `xm.distance < 0.5`). For custom matching radii or catalogs not in the pre-computed list, use `q3c_join` directly as shown in Section 6.

> ⚠️ **Warning:** The column names for external catalogs (e.g., `w.source_id`, `v.sourceid`) depend on the specific catalog schema. Use Data Lab's schema browser to verify column names before constructing your queries.